# K-Means From Scratch

Wiki reference for [the K-Means algorithm](https://ml-viz-ruby.vercel.app/wiki/kmeans-algorithm).

**The idea in one sentence.** K-Means alternates **assign** (each point to its nearest centroid)
and **update** (each centroid to its points' mean) — Lloyd's algorithm — which monotonically
**decreases inertia** (within-cluster sum of squares) to a *local* optimum that depends on the
random initialization.

We implement K-Means from scratch, **validate the monotone inertia descent and cluster
recovery**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch K-Means

In [ ]:
def kmeans(X, K, n_iter=200, seed=42):
    rng = np.random.default_rng(seed)
    mu = X[rng.choice(len(X), K, replace=False)]
    history = []

    for it in range(n_iter):
        # Step 1 — assign
        dists = np.linalg.norm(X[:,None] - mu[None], axis=2)  # (n, K)
        labels = dists.argmin(axis=1)

        # Step 2 — update
        new_mu = np.array([X[labels==k].mean(axis=0) if (labels==k).any() else mu[k]
                           for k in range(K)])

        J = sum(np.sum((X[labels==k]-new_mu[k])**2) for k in range(K))
        history.append(J)

        if np.allclose(new_mu, mu):
            print(f"Converged at iteration {it+1}")
            break
        mu = new_mu

    return labels, mu, history

# Wiki worked example (5 points, K=2)
X5 = np.array([[1,1],[1.5,2],[3,4],[5,7],[3.5,5]], dtype=float)
labels5, mu5, hist5 = kmeans(X5, K=2, seed=0)
print("Labels:", labels5)
print("Centroids:", mu5.round(2))
print("Final J:", round(hist5[-1], 3))

## Visualize convergence on a larger dataset

In [ ]:
rng = np.random.default_rng(0)
X = np.vstack([rng.normal([0,0],1,(100,2)),
               rng.normal([5,5],1.2,(100,2)),
               rng.normal([0,6],0.8,(80,2))])

K = 3
labels, mu, history = kmeans(X, K=K)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
colors = ['#6366f1','#f59e0b','#10b981']
for k in range(K):
    axes[0].scatter(X[labels==k,0], X[labels==k,1], color=colors[k], s=15, alpha=0.6)
axes[0].scatter(mu[:,0], mu[:,1], c='white', marker='X', s=200, zorder=5)
axes[0].set_title(f'K-Means K={K}')

axes[1].plot(history, color='#6366f1')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Inertia J')
axes[1].set_title('Objective (inertia) vs iteration')
plt.tight_layout(); plt.show()

### Validate: inertia decreases monotonically and clusters are recovered

Each assign+update step can only lower (or hold) the within-cluster sum of squares, so the
inertia history is non-increasing — and on three well-separated blobs the centroids land near
the true centers. We confirm both.

In [ ]:
print('inertia history:', [round(h, 1) for h in history])
assert all(history[i] >= history[i+1] - 1e-9 for i in range(len(history) - 1)), 'K-Means inertia never increases (Lloyd descent)'
for c in [[0, 0], [5, 5], [0, 6]]:
    assert min(np.linalg.norm(mu - np.array(c), axis=1)) < 1.5, 'a centroid recovers each true cluster center'
print('\n✅ Lloyd’s algorithm monotonically reduces inertia and finds the three clusters')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **local optima** | random init changes the result (demo) — use n_init / k-means++ |
| **choosing K** | inertia always falls with K; use the elbow / silhouette |
| **assumes spherical clusters** | fails on elongated / non-convex shapes (see DBSCAN) |
| **sensitive to scale** | standardize features first |
| **outliers** | pull centroids; consider k-medoids |

Demo: different initialisations reach different local optima.

In [ ]:
# K-Means minimises a NON-convex objective, so the random initialisation matters: different
# seeds land in different local optima. (On these 3 well-separated blobs it usually finds the
# global optimum; asking for MORE clusters than exist, k=5, makes the landscape bumpy and the
# inits disagree.) That is why real implementations run n_init times and keep the best.
finals = [kmeans(X, K=5, seed=s)[2][-1] for s in range(20)]
finals = np.array(finals)
n_bad = int((finals > finals.min() * 1.05).sum())
print(f'final inertia over 20 inits (k=5): best {finals.min():.1f}, worst {finals.max():.1f}; {n_bad} landed >5% above best')
assert finals.max() > finals.min() + 1e-6, 'different initialisations reach different local optima'
print('\nK-Means is not convex -> run several inits (n_init) and keep the lowest-inertia result.')

## ✏️ Your turn

**Task:** Implement the **elbow method** — run K-Means for K=1..8 on the dataset above and plot inertia vs K. Identify the elbow.

**Extension:** Implement **K-Means++** initialization: pick the first centroid randomly, then each subsequent centroid with probability proportional to squared distance from the nearest existing centroid.

In [ ]:
# TODO(you): run kmeans for K in range(1, 9) and collect final inertias
# inertias = []
# for k in range(1, 9):
#     _, _, hist = kmeans(X, K=k)
#     inertias.append(hist[-1])

In [ ]:
# assert len(inertias) == 8
# assert inertias[0] > inertias[-1], 'inertia should decrease with K'

<details><summary>Solution</summary>

```python
inertias = []
for k in range(1, 9):
    _, _, hist = kmeans(X, K=k)
    inertias.append(hist[-1])

plt.figure(figsize=(7,4))
plt.plot(range(1,9), inertias, 'o-', color='#6366f1')
plt.axvline(3, color='#f59e0b', linestyle='--', label='Elbow at K=3')
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow method')
plt.legend(); plt.show()
# Elbow at K=3 (matching the 3 true clusters)
```
</details>

## Key takeaways

- **Assign + update (Lloyd's):** the two-step loop at the heart of K-Means.
- **Monotone inertia:** each step lowers (or holds) the within-cluster sum of squares (verified).
- **Recovers well-separated clusters** (verified) — but the objective is **non-convex**.
- **Init matters:** different seeds reach different local optima (demo) — use `n_init` (and
  k-means++).